In [1]:
# Install the Hugging Face datasets library if you don't have it
!pip install datasets pandas

In [2]:
import pandas as pd
from datasets import load_dataset
import json
import os

In [3]:
print("Loading Databricks Dolly 15k dataset...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

# Convert to a Pandas DataFrame for easier manipulation
df = dataset.to_pandas()
print(f"Original dataset size: {len(df)} rows")

# Define how many rows you want to manually annotate to start
# (You can change this number. 1000 is a good first milestone)
SAMPLE_SIZE = 1000

# Shuffle and sample the dataset
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42)

# Select only the columns needed for context and the target instruction
annotation_df = sample_df[['category', 'context', 'instruction']].copy()

# Create the empty column for your manual unoptimized prompts
annotation_df['unoptimized_instruction'] = ""

# Reorder columns to make annotation easier (put the blank column right next to the original)
annotation_df = annotation_df[['category', 'context', 'instruction', 'unoptimized_instruction']]

# Export to CSV
csv_filename = "dolly_to_annotate.csv"
annotation_df.to_csv(csv_filename, index=False)

print(f"\nSuccess! Exported {SAMPLE_SIZE} rows to '{csv_filename}'.")
print("Download this file, open it in Excel or Google Sheets, and start annotating.")

Loading Databricks Dolly 15k dataset...


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Original dataset size: 15011 rows

Success! Exported 1000 rows to 'dolly_to_annotate.csv'.
Download this file, open it in Excel or Google Sheets, and start annotating.


In [3]:
from datasets import load_dataset

alpaca = load_dataset("yahma/alpaca-cleaned", split="train")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")

print(alpaca[0])
print(dolly[0])

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Austra

In [4]:
from collections import Counter

# ---- Attributes / schema ----
print("Alpaca-cleaned features:", alpaca.features)
print("Dolly features:", dolly.features)

# ---- Column names only, quick view ----
print("Alpaca columns:", alpaca.column_names)
print("Dolly columns:", dolly.column_names)

# ---- Dolly: full category breakdown with counts ----
dolly_categories = Counter(row["category"] for row in dolly)
print("\nDolly category distribution:")
for cat, count in dolly_categories.most_common():
    print(f"  {cat}: {count} ({count/len(dolly)*100:.1f}%)")

# ---- Alpaca has no category field — confirm it ----
print("\nAlpaca has no 'category' column — confirmed by column_names above.")
print("Alpaca total rows:", len(alpaca))
print("Dolly total rows:", len(dolly))

Alpaca-cleaned features: {'output': Value('string'), 'input': Value('string'), 'instruction': Value('string')}
Dolly features: {'instruction': Value('string'), 'context': Value('string'), 'response': Value('string'), 'category': Value('string')}
Alpaca columns: ['output', 'input', 'instruction']
Dolly columns: ['instruction', 'context', 'response', 'category']

Dolly category distribution:
  open_qa: 3742 (24.9%)
  general_qa: 2191 (14.6%)
  classification: 2136 (14.2%)
  closed_qa: 1773 (11.8%)
  brainstorming: 1766 (11.8%)
  information_extraction: 1506 (10.0%)
  summarization: 1188 (7.9%)
  creative_writing: 709 (4.7%)

Alpaca has no 'category' column — confirmed by column_names above.
Alpaca total rows: 51760
Dolly total rows: 15011


In [5]:
# Same breakdown as a sorted table, easiest to paste into your report
import pandas as pd
dolly_df = pd.DataFrame(dolly)
print(dolly_df['category'].value_counts())
print(dolly_df['category'].value_counts(normalize=True) * 100)

category
open_qa                   3742
general_qa                2191
classification            2136
closed_qa                 1773
brainstorming             1766
information_extraction    1506
summarization             1188
creative_writing           709
Name: count, dtype: int64
category
open_qa                   24.928386
general_qa                14.595963
classification            14.229565
closed_qa                 11.811338
brainstorming             11.764706
information_extraction    10.032643
summarization              7.914196
creative_writing           4.723203
Name: proportion, dtype: float64
